# Análisis de comportamiento de clientes en retail online

## 🎯 Objetivo

Analizar el comportamiento de los clientes en un entorno de retail online para identificar patrones de compra, detectar oportunidades de mejora y apoyar la toma de decisiones estratégicas.

## 🧠 Contexto

El dataset contiene transacciones reales de una empresa de retail online en el Reino Unido. Cada registro representa un producto dentro de una compra, incluyendo información como cantidad, precio, cliente y fecha.

Este análisis busca entender cómo compran los clientes, qué factores influyen en el valor de compra y cómo se pueden detectar comportamientos relevantes para el negocio.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
# Cargamos los datos
df = pd.read_excel('Online Retail.xlsx')

df.head()

In [ ]:
# Revisamos la información general
df.info()

No se econtraron tipos de datos incorrectos

## 🫧 Limpieza de datos

In [ ]:
# Función para convertir nombres de columnas a snake_case
def to_snake_case(col_name):
    col_name = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', col_name)
    col_name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', col_name)
    return col_name.lower()

# Función para limpiar el DataFrame
def clean_dataframe(df):
    df = df.rename(columns=lambda x: to_snake_case(x))

    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

    return df

# Aplicamos limpieza inicial
df = clean_dataframe(df)

display(df.head())

In [ ]:
# Revisamos valores ausentes
print(df.isna().sum())

In [ ]:
# Revisamos filas duplicadas
print(df.duplicated().sum())

### 📋 Observaciones

Se encontraron valores ausentes en `description` y `customer_id`. En el caso de `description`, la cantidad es muy baja, por lo que no debería afectar de forma relevante el análisis.

Sin embargo, `customer_id` presenta una cantidad considerable de valores nulos, lo que indica que no todas las transacciones están asociadas a un cliente identificable. Esto limita cualquier análisis a nivel de usuario.

Por esta razón, el análisis se va a abordar de dos formas: por un lado, se utilizará todo el dataset para entender el comportamiento general de las transacciones, y por otro, cuando se requiera analizar clientes, se trabajará únicamente con los registros que sí cuentan con `customer_id`.

Además, se detectaron filas duplicadas, por lo que se procederá a eliminarlas para evitar distorsiones en los resultados.

También, es importante revisar los valores de `quantity` y `unit_price`, ya que en este tipo de datasets pueden aparecer devoluciones, cancelaciones o registros atípicos que no representan compras válidas. Si se incluyen sin control, podrían distorsionar tanto los ingresos como el comportamiento de compra.

In [ ]:
# Eliminamos filas duplicadas
df = df.drop_duplicates()

# Comprobamos que ya no queden duplicados
print(df.duplicated().sum())

In [ ]:
# Revisamos registros con cantidades menores o iguales a cero
print((df['quantity'] <= 0).sum())

# Revisamos registros con precios menores o iguales a cero
print((df['unit_price'] <= 0).sum())

### 📋 Observaciones

Se identificaron registros con cantidades o precios menores o iguales a cero. En un contexto de retail online, estos casos no representan compras válidas, por lo que podrían corresponder a devoluciones, cancelaciones o inconsistencias en el registro.

Como el objetivo de este proyecto es analizar el comportamiento de compra y el valor generado por los clientes, estos registros se eliminarán para trabajar únicamente con transacciones válidas.

Además, se observa que la cantidad de registros con `quantity <= 0` es considerablemente mayor que aquellos con `unit_price <= 0`, lo que sugiere que una parte importante de estos casos podría estar asociada a devoluciones de productos más que a errores de precio.

In [ ]:
# Conservamos únicamente transacciones válidas
df = df[df['quantity'] > 0]
df = df[df['unit_price'] > 0]

## 🔎 Análisis exploratorio de datos

### Creación de la variable de ingreso

Para poder analizar el valor económico de las transacciones, se creará la columna `revenue`, calculada como el producto de `quantity` por `unit_price`.

Esta variable permitirá comparar clientes, identificar patrones de compra y evaluar qué tipo de comportamiento está más relacionado con una mayor generación de ingresos.

In [ ]:
# Calculamos el ingreso generado por cada registro
df['revenue'] = df['quantity'] * df['unit_price']

df.head()

### Construcción de métricas a nivel cliente

Para analizar el comportamiento de los usuarios, es necesario transformar el dataset de transacciones en un nivel agregado por cliente.

Se calcularán métricas clave como el número de órdenes, la cantidad total comprada, el ingreso generado y el rango de fechas de compra, con el objetivo de entender mejor los patrones de consumo de cada cliente.

In [ ]:
# Creamos dataset a nivel cliente
customer_df = df.groupby('customer_id').agg({
    'invoice_no': 'nunique',
    'quantity': 'sum',
    'revenue': 'sum',
    'invoice_date': ['min', 'max']
})

# Aplanamos nombres de columnas
customer_df.columns = [
    'num_orders',
    'total_quantity',
    'total_revenue',
    'first_purchase',
    'last_purchase'
]

customer_df = customer_df.reset_index()

customer_df.head()

### Creación de variable de recencia

Para entender qué tan recientes son las interacciones de cada cliente, se calculará la recencia como la cantidad de días desde su última compra hasta la fecha más reciente del dataset.

Esta métrica permitirá identificar clientes activos frente a aquellos que podrían estar en riesgo de abandono.

In [ ]:
# Calculamos la recencia (días desde la última compra hasta la fecha más reciente del dataset)

latest_date = customer_df['last_purchase'].max()

customer_df['recency_days'] = (latest_date - customer_df['last_purchase']).dt.days

customer_df.head()

### Análisis de comportamiento de clientes

A partir de las métricas construidas (recencia, número de órdenes e ingreso total), se analizará la distribución del comportamiento de los clientes para identificar patrones relevantes.

In [ ]:
# Distribución de recencia
sns.histplot(customer_df['recency_days'], bins=50)
plt.title('Distribución de recencia de clientes')
plt.xlabel('Días desde la última compra')
plt.ylabel('Número de clientes')
plt.show()

### 📋 Observaciones

La distribución de la recencia muestra una alta concentración de clientes con valores bajos, lo que indica que una parte importante de los usuarios ha realizado compras recientemente.

Sin embargo, también se observa una cola larga hacia valores altos, lo que sugiere la existencia de clientes que no han interactuado con la plataforma en un periodo considerable de tiempo.

Este comportamiento indica que, aunque hay una base activa de clientes, también existe un segmento relevante que podría estar en riesgo de abandono.

### Análisis de ingresos por cliente

Se analizará la distribución del ingreso total generado por los clientes con el objetivo de identificar la concentración del valor económico y detectar posibles clientes de alto valor.

In [ ]:
sns.histplot(customer_df['total_revenue'], bins=50)
plt.title('Distribución de ingresos por cliente')
plt.xlabel('Ingreso total')
plt.ylabel('Número de clientes')
plt.show()

### Ajuste de escala en la distribución de ingresos

Dado que la distribución del ingreso por cliente presenta valores extremadamente altos en comparación con la mayoría de los registros, se aplicará una transformación logarítmica en el eje horizontal.

Esto permitirá visualizar con mayor claridad la dispersión de los datos y analizar de forma más precisa el comportamiento de la mayoría de los clientes, evitando que los valores atípicos dominen la gráfica.

In [ ]:
sns.histplot(customer_df['total_revenue'], bins=50)
plt.xscale('log')
plt.title('Distribución de ingresos por cliente (escala logarítmica)')
plt.xlabel('Ingreso total (escala log)')
plt.ylabel('Número de clientes')
plt.show()

### 📋 Observaciones

Al aplicar la escala logarítmica, se confirma que la distribución del ingreso por cliente es altamente asimétrica.

La mayoría de los clientes se concentra en niveles bajos de ingreso, mientras que existe una cola larga de clientes con valores significativamente mayores.

Esto sugiere que una proporción reducida de clientes concentra gran parte del valor económico, lo cual es consistente con dinámicas típicas de negocio donde unos pocos clientes generan la mayor parte de los ingresos.

Este hallazgo refuerza la necesidad de segmentar a los clientes para identificar y analizar aquellos de alto valor.

### Distribución de número de órdenes por cliente

Se analizará la distribución del número de órdenes realizadas por cliente, con el objetivo de entender la frecuencia de compra y detectar posibles diferencias entre clientes ocasionales y recurrentes.

In [ ]:
sns.histplot(customer_df['num_orders'], bins=50)
plt.title('Distribución de número de órdenes por cliente')
plt.xlabel('Número de órdenes')
plt.ylabel('Número de clientes')
plt.show()

### 📋 Observaciones

La distribución del número de órdenes por cliente presenta una fuerte asimetría positiva, donde la mayoría de los clientes realiza pocas compras, mientras que un grupo reducido concentra una mayor frecuencia de transacciones.

Esto indica que gran parte de los clientes tiene un comportamiento ocasional, mientras que existe un segmento más pequeño de clientes recurrentes que interactúan con mayor frecuencia.

Este patrón sugiere la importancia de identificar y entender a los clientes más activos, ya que podrían representar una fuente relevante de ingresos sostenidos para el negocio.

### Relación entre número de órdenes e ingreso total

Se analizará la relación entre la frecuencia de compra y el ingreso generado por cliente, con el objetivo de identificar si los clientes que compran más también generan mayor valor económico.

In [ ]:
sns.scatterplot(x=customer_df['num_orders'], y=customer_df['total_revenue'])
plt.title('Relación entre número de órdenes e ingreso total')
plt.xlabel('Número de órdenes')
plt.ylabel('Ingreso total')
plt.show()

### 📋 Observaciones

La relación entre el número de órdenes y el ingreso total no muestra una tendencia lineal clara, lo que indica que una mayor frecuencia de compra no necesariamente implica un mayor valor económico por cliente.

Se identifican distintos tipos de comportamiento: clientes que realizan pocas compras pero generan altos ingresos, así como clientes que compran con mayor frecuencia pero con menor valor por transacción.

Además, se observan algunos valores atípicos que representan clientes con un impacto significativamente alto en los ingresos totales.

Este comportamiento refuerza la importancia de analizar conjuntamente la frecuencia y el valor monetario para comprender el verdadero aporte de cada cliente al negocio.

### Relación entre frecuencia, ingreso y recencia

Se incorporará la variable de recencia en la visualización para analizar cómo el tiempo desde la última compra se relaciona con la frecuencia y el ingreso generado por los clientes, permitiendo identificar patrones de actividad y valor.

In [ ]:
sns.scatterplot(
    x=customer_df['num_orders'],
    y=customer_df['total_revenue'],
    hue=customer_df['recency_days'],
    palette='viridis'
)

plt.title('Relación entre número de órdenes, ingreso y recencia')
plt.xlabel('Número de órdenes')
plt.ylabel('Ingreso total')
plt.show()

### 📋 Observaciones

Al incorporar la variable de recencia en la visualización, se identifican patrones más complejos en el comportamiento de los clientes.

Se observa que algunos clientes con altos niveles de ingreso presentan valores bajos de recencia, lo que indica que siguen siendo activos y representan un segmento de alto valor para el negocio.

Sin embargo, también se identifican clientes que, a pesar de haber generado ingresos elevados, muestran altos valores de recencia, lo que sugiere que no han realizado compras recientemente y podrían encontrarse en riesgo de abandono.

Asimismo, existen clientes con alta frecuencia de compra pero bajo ingreso total, lo que indica un menor valor por transacción.

Estos hallazgos refuerzan la importancia de analizar conjuntamente la recencia, la frecuencia y el valor monetario para identificar segmentos estratégicos de clientes y definir posibles acciones de negocio.

### Construcción de scores RFM

Con el fin de segmentar a los clientes según su comportamiento, se asignarán puntajes a las métricas de recencia, frecuencia y valor monetario.

Para ello, cada variable se dividirá en quintiles. En el caso de la recencia, los clientes con menos días desde su última compra recibirán un puntaje más alto, mientras que para frecuencia e ingreso, los valores más altos recibirán mejores puntuaciones.

Posteriormente, los puntajes individuales se combinarán en un único score RFM, el cual permitirá tener una representación más completa del comportamiento de cada cliente.

Esta segmentación permitirá identificar distintos perfiles de clientes, como usuarios de alto valor, clientes leales o clientes en riesgo de abandono.

In [ ]:
# Creamos score de recencia
# Menor recency_days = mejor score (clientes más recientes son más valiosos)
customer_df['r_score'] = pd.qcut(
    customer_df['recency_days'],
    q=5,
    labels=[5, 4, 3, 2, 1]  # invertido porque menor = mejor
).astype(int)

# Creamos score de frecuencia
# Mayor número de órdenes = mejor score (usamos rank para evitar problemas con valores repetidos en qcut)
customer_df['f_score'] = pd.qcut(
    customer_df['num_orders'].rank(method='first'),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Creamos score monetario
# Mayor ingreso total = mejor score (mismo ajuste por duplicados)
customer_df['m_score'] = pd.qcut(
    customer_df['total_revenue'].rank(method='first'),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Creamos score RFM combinado
# Concatenamos los scores para tener una representación completa del comportamiento del cliente
customer_df['rfm_score'] = (
    customer_df['r_score'].astype(str) +
    customer_df['f_score'].astype(str) +
    customer_df['m_score'].astype(str)
)

# Visualizamos resultados
customer_df.head()

## Segmentación de clientes basada en RFM

Con base en los scores de recencia, frecuencia y valor monetario, se segmentará a los clientes en distintos grupos que permitan interpretar su comportamiento.

Estos segmentos facilitarán identificar clientes de alto valor, usuarios leales, así como clientes en riesgo de abandono, permitiendo orientar estrategias de negocio más efectivas.

### Criterio de segmentación

La segmentación se basa en la lógica de RFM, donde se combinan tres dimensiones clave del comportamiento del cliente:

- **Recencia (R):** qué tan reciente fue la última compra  
- **Frecuencia (F):** qué tan seguido compra el cliente  
- **Valor monetario (M):** cuánto dinero genera  

A partir de esto, se definen los siguientes criterios:

- **High Value:** clientes con puntuaciones altas en las tres dimensiones (compran recientemente, con frecuencia y generan alto ingreso)  
- **Loyal:** clientes con alta frecuencia de compra, independientemente de su recencia o gasto  
- **At Risk:** clientes con baja recencia, es decir, que llevan mucho tiempo sin comprar  
- **Regular:** clientes que no cumplen con los criterios anteriores  

Esta clasificación permite diferenciar perfiles de clientes y entender mejor su comportamiento para futuras estrategias.

In [ ]:
# Función para segmentar clientes según RFM
def segment_customer(row):

    # Clientes top: alto en todo
    if row['r_score'] >= 4 and row['f_score'] >= 4 and row['m_score'] >= 4:
        return 'High Value'

    # Clientes frecuentes
    elif row['f_score'] >= 4:
        return 'Loyal'

    # Clientes que llevan tiempo sin comprar
    elif row['r_score'] <= 2:
        return 'At Risk'

    # Resto de clientes
    else:
        return 'Regular'

# Aplicamos segmentación
customer_df['segment'] = customer_df.apply(segment_customer, axis=1)

# Revisamos resultados
customer_df['segment'].value_counts()

### 📋 Observaciones

Se observa que el segmento más grande corresponde a clientes en riesgo (At Risk), lo que indica que una proporción importante de usuarios lleva tiempo sin realizar compras.

Por otro lado, existe un grupo relevante de clientes de alto valor (High Value), los cuales combinan alta frecuencia, recencia e ingreso, representando una base clave para el negocio.

Los clientes leales (Loyal) también destacan por su frecuencia de compra, aunque no necesariamente presentan los niveles más altos de ingreso o recencia.

Finalmente, los clientes regulares constituyen un grupo intermedio, con comportamientos menos definidos, lo que sugiere oportunidades para incentivar su actividad y migrarlos hacia segmentos de mayor valor.

### Análisis de valor por segmento

Con el fin de entender el impacto económico de cada grupo de clientes, se analizará el ingreso total generado por cada segmento, así como el ingreso promedio por cliente.

Esto permitirá identificar qué segmentos aportan mayor valor al negocio y cuáles representan oportunidades de crecimiento o mejora.

In [ ]:
# Ingreso total por segmento
customer_df.groupby('segment')['total_revenue'].sum().sort_values(ascending=False)

In [ ]:
# Ingreso promedio por cliente por segmento
customer_df.groupby('segment')['total_revenue'].mean().sort_values(ascending=False)

### 📋 Observaciones

Se observa una clara concentración del valor económico en el segmento de clientes de alto valor (High Value), los cuales generan una proporción significativamente mayor de ingresos en comparación con los demás segmentos.

Además, este grupo no solo destaca en el ingreso total, sino también en el ingreso promedio por cliente, lo que indica que su comportamiento combina frecuencia, recencia y alto gasto.

Por otro lado, los clientes leales (Loyal) presentan un aporte relevante, aunque considerablemente menor, lo que sugiere que representan una base importante pero con menor valor individual.

Finalmente, los segmentos Regular y At Risk muestran los niveles más bajos de ingreso promedio, lo que indica que, a pesar de su tamaño, su contribución económica es limitada.

Este análisis evidencia una distribución desigual del valor, donde un grupo relativamente reducido de clientes concentra la mayor parte de los ingresos, lo cual es consistente con el principio de Pareto.

## Conclusión

El análisis muestra una fuerte concentración del valor en el segmento High Value, el cual genera más de 5.7 millones en ingresos, superando ampliamente al resto de los segmentos. Esto indica que el negocio depende en gran medida de un grupo reducido de clientes, lo que representa tanto una fortaleza como un riesgo.

Por otro lado, el segmento At Risk es el más numeroso, lo que sugiere que una proporción importante de clientes ha dejado de interactuar recientemente. Esto representa una oportunidad clara de reactivación, ya que incluso una recuperación parcial podría impactar de forma relevante en los ingresos.

Asimismo, se observó que la frecuencia de compra no siempre se traduce en mayor ingreso, lo que indica la existencia de distintos perfiles de clientes: algunos realizan pocas compras pero de alto valor, mientras que otros compran con mayor frecuencia pero con compras de menor valor.

En este contexto, se pueden identificar algunas acciones clave para el negocio. Por un lado, resulta fundamental implementar estrategias de retención para los clientes de alto valor, dado su impacto en los ingresos. Por otro, el segmento en riesgo representa una oportunidad para campañas de reactivación, como incentivos o recordatorios personalizados.

Finalmente, para los clientes frecuentes pero de menor valor, podrían explorarse estrategias orientadas a incrementar el valor promedio por compra, mientras que los clientes regulares representan un segmento con potencial de crecimiento mediante acciones de fidelización.